## Latest energy-conserving vs entropy hu-constant CSV profiles

This section reads the most recent energy-conserving CSV profile and the most recent entropy/`hu_const` CSV profile from `csv_share/`, compares their common profile columns, and visualizes both the profiles and their pointwise differences. It uses only the CSV files and does not rerun any solver code.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

csv_dir = Path("csv_share")


def _latest_profile_csv(pattern):
    candidates = sorted(csv_dir.glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No CSV files matched {csv_dir / pattern}")
    non_copy = [path for path in candidates if " copy" not in path.stem.lower()]
    if non_copy:
        candidates = non_copy
    return max(candidates, key=lambda path: path.stat().st_mtime)


energy_csv_path = _latest_profile_csv("*energy_conserving*profiles*.csv")
hu_const_csv_path = _latest_profile_csv("*hu_const*profiles*.csv")

raw_profiles = {
    "energy conserving": pd.read_csv(energy_csv_path),
    "entropy hu_const": pd.read_csv(hu_const_csv_path),
}

# The first row is the nuclear-side x=0^- state in these profile CSVs. For
# profile-to-profile comparison, keep only the quark-side rows with finite D.
profiles = {
    label: df.loc[np.isfinite(df["D"])].copy().reset_index(drop=True)
    for label, df in raw_profiles.items()
}

selected_profile_files = pd.DataFrame(
    [
        {
            "label": "energy conserving",
            "path": str(energy_csv_path),
            "modified": pd.Timestamp.fromtimestamp(energy_csv_path.stat().st_mtime),
            "raw_rows": len(raw_profiles["energy conserving"]),
            "quark_rows": len(profiles["energy conserving"]),
        },
        {
            "label": "entropy hu_const",
            "path": str(hu_const_csv_path),
            "modified": pd.Timestamp.fromtimestamp(hu_const_csv_path.stat().st_mtime),
            "raw_rows": len(raw_profiles["entropy hu_const"]),
            "quark_rows": len(profiles["entropy hu_const"]),
        },
    ]
)

display(selected_profile_files)


## Profile Column Selection

Contains the `PROFILE_COLUMN_SELECTION` notebook section.


In [ ]:
preferred_columns = [
    "a",
    "u",
    "nB",
    "nK",
    "muB",
    "muK",
    "T",
    "P",
    "e",
    "h",
    "h_over_nB",
    "P_over_e",
    "D",
    "gamma",
    "eta",
]

common_columns = [
    column
    for column in preferred_columns
    if all(column in df.columns for df in profiles.values())
]

energy_profile = profiles["energy conserving"]
hu_profile = profiles["entropy hu_const"]
energy_x = energy_profile["x"].to_numpy(dtype=float)
hu_x = hu_profile["x"].to_numpy(dtype=float)

if not np.all(np.diff(energy_x) > 0.0):
    raise RuntimeError("Energy-conserving profile x values are not strictly increasing after dropping x=0^- row")
if not np.all(np.diff(hu_x) > 0.0):
    raise RuntimeError("hu_const profile x values are not strictly increasing after dropping x=0^- row")

comparison_rows = []
for column in common_columns:
    energy_values = energy_profile[column].to_numpy(dtype=float)
    hu_values = hu_profile[column].to_numpy(dtype=float)
    hu_on_energy_x = np.interp(energy_x, hu_x, hu_values)
    diff = energy_values - hu_on_energy_x
    finite_scale = np.abs(hu_on_energy_x[np.isfinite(hu_on_energy_x)])
    floor = max(float(np.nanmedian(finite_scale)) if finite_scale.size else 0.0, np.finfo(float).tiny)
    rel_diff = diff / np.maximum(np.abs(hu_on_energy_x), floor)
    comparison_rows.append(
        {
            "quantity": column,
            "energy_min": float(np.nanmin(energy_values)),
            "energy_max": float(np.nanmax(energy_values)),
            "hu_const_min": float(np.nanmin(hu_values)),
            "hu_const_max": float(np.nanmax(hu_values)),
            "max_abs_diff_on_energy_x": float(np.nanmax(np.abs(diff))),
            "median_abs_rel_diff_on_energy_x": float(np.nanmedian(np.abs(rel_diff))),
            "max_abs_rel_diff_on_energy_x": float(np.nanmax(np.abs(rel_diff))),
        }
    )

profile_comparison_summary = pd.DataFrame(comparison_rows)
display(profile_comparison_summary)


## Profile Numerical Helpers

Contains the `PROFILE_NUMERICAL_HELPERS` notebook section.


In [ ]:
import math
import matplotlib.pyplot as plt

plot_columns = common_columns
x_scale = 1.0e6

ncols = 3
nrows = math.ceil(len(plot_columns) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.0 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, column in zip(axes, plot_columns):
    ax.plot(
        energy_profile["x"].to_numpy(dtype=float) / x_scale,
        energy_profile[column].to_numpy(dtype=float),
        lw=2.0,
        label="energy conserving",
    )
    ax.plot(
        hu_profile["x"].to_numpy(dtype=float) / x_scale,
        hu_profile[column].to_numpy(dtype=float),
        lw=1.8,
        ls="--",
        label="entropy hu_const",
    )
    ax.set_title(column)
    ax.set_xlabel(r"$x$ [$10^6$]")
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)

for ax in axes[len(plot_columns):]:
    ax.set_visible(False)

fig.suptitle("Latest CSV profile comparison", y=1.01)
plt.show()

nrows = math.ceil(len(plot_columns) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.0 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, column in zip(axes, plot_columns):
    energy_values = energy_profile[column].to_numpy(dtype=float)
    hu_on_energy_x = np.interp(
        energy_x,
        hu_x,
        hu_profile[column].to_numpy(dtype=float),
    )
    diff = energy_values - hu_on_energy_x
    ax.plot(energy_x / x_scale, diff, lw=1.8, color="tab:purple")
    ax.axhline(0.0, color="0.45", lw=1.0, ls="--")
    ax.set_title(f"energy - hu_const: {column}")
    ax.set_xlabel(r"$x$ [$10^6$]")
    ax.grid(alpha=0.25)

for ax in axes[len(plot_columns):]:
    ax.set_visible(False)

fig.suptitle("Pointwise difference after interpolating hu_const onto energy-conserving x-grid", y=1.01)
plt.show()


## Profile Data Loading

Contains the `PROFILE_DATA_LOADING` notebook section.


In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-cache-26su4")

import numpy as np
import pandas as pd
from IPython.display import display

import RMFsolver.constants as const
import RMFsolver.RMFparameter as para
from RMFsolver import phase_velocity as pv

csv_path = Path("csv_share/energy_conserving_profiles_T10MeV_B14_180MeV_nBN_3n0_aQstar_0p3.csv")
reference_all = pd.read_csv(csv_path)
reference_profile = reference_all.loc[np.isfinite(reference_all["D"])].copy().reset_index(drop=True)
reference_endpoint = reference_all.loc[~np.isfinite(reference_all["D"])].copy().reset_index(drop=True)

T_N = 10.0
B_one_forth = 180.0
aQstar = 0.3
nB_N = 3.0 * 0.16 * const.MeV_fm**3
tail_eps = min(float(reference_profile["a"].iloc[-1] / aQstar), 7.0e-9)
jB_guess = float(np.median(reference_profile["u"].to_numpy() * reference_profile["nB"].to_numpy()))
TQ_guess = float(reference_profile["T"].iloc[-1])
n_mesh = int(min(max(80, len(reference_profile)), 240))
tol_bvp = 1.0e-6

result = pv.solve_front_energy_conserving(
    T=T_N,
    nB_N=nB_N,
    B_one_forth=B_one_forth,
    aQstar=aQstar,
    ms=0.0,
    param=para.paraQMCRMF3,
    NM_type="PNM",
    tail_eps=tail_eps,
    n_mesh=n_mesh,
    tol_bvp=tol_bvp,
    max_nodes=50000,
    jB_guess=jB_guess,
    TQ_guess=TQ_guess,
    jB_bounds=(1.0e-5, 5.0),
    return_profile=True,
    verb=False,
)

if not bool(result.get("success")):
    raise RuntimeError(result.get("message", "full energy-conserving solve failed"))

new_profile = pd.DataFrame(
    {
        "x": np.asarray(result["x"], dtype=float),
        "a": np.asarray(result["a"], dtype=float),
        "q": np.asarray(result["q"], dtype=float),
        "u": np.asarray(result["u"], dtype=float),
        "nB": np.asarray(result["nB"], dtype=float),
        "muB": np.asarray(result["muB"], dtype=float),
        "muK": np.asarray(result["muK"], dtype=float),
        "T": np.asarray(result["T_profile"], dtype=float),
        "h": np.asarray(result["h_profile"], dtype=float),
        "D": np.asarray(result["D_profile"], dtype=float),
        "gamma": np.asarray(result["gamma_profile"], dtype=float),
        "eta": np.asarray(result["eta_profile"], dtype=float),
    }
)
new_profile["P"] = float(result["Pi"]) - new_profile["h"] * new_profile["u"]**2
new_profile["h_over_nB"] = new_profile["h"] / new_profile["nB"]

comparison_columns = ["a", "u", "nB", "muB", "muK", "T", "h_over_nB", "D", "gamma", "eta"]
comparison_rows = []
ref_x = reference_profile["x"].to_numpy(dtype=float)
new_x = new_profile["x"].to_numpy(dtype=float)
for column in comparison_columns:
    ref_values = reference_profile[column].to_numpy(dtype=float)
    new_values = np.interp(ref_x, new_x, new_profile[column].to_numpy(dtype=float))
    diff = new_values - ref_values
    scale = max(float(np.nanmax(np.abs(ref_values))), np.finfo(float).tiny)
    comparison_rows.append(
        {
            "quantity": column,
            "max_abs_diff": float(np.nanmax(np.abs(diff))),
            "max_rel_diff_to_saved_range": float(np.nanmax(np.abs(diff)) / scale),
            "saved_start": float(ref_values[0]),
            "new_start": float(new_values[0]),
            "saved_end": float(ref_values[-1]),
            "new_end": float(new_values[-1]),
        }
    )

comparison = pd.DataFrame(comparison_rows)
diagnostics = pd.Series(
    {
        "csv_path": str(csv_path),
        "solver_call": "pv.solve_front_energy_conserving",
        "success": bool(result["success"]),
        "message": result["message"],
        "tol_bvp": tol_bvp,
        "tail_eps": tail_eps,
        "n_mesh": n_mesh,
        "bvp_nodes": int(result["bvp_nodes"]),
        "bvp_niter": int(result["bvp_niter"]),
        "jB_guess_from_csv": jB_guess,
        "jB_new": float(result["jB"]),
        "TQ_guess_from_csv_tail": TQ_guess,
        "T_Q_new": float(result["T_Q"]),
        "T_Qstar_new": float(result["T_Qstar"]),
        "q_left_residual_norm": float(result["q_left_residual_norm"]),
        "tail_residual_norm": float(result["tail_residual_norm"]),
        "E_right_residual_norm": float(result["E_right_residual_norm"]),
        "kaon_equation_residual_norm": float(result.get("kaon_equation_residual_norm", np.nan)),
    },
    name="value",
)

display(diagnostics.to_frame())
display(comparison)


## Profile Plotting Setup

Visualizes the outputs from the `PROFILE_PLOTTING_SETUP` workflow section.


In [ ]:
import matplotlib.pyplot as plt

plot_specs = [
    ("a", r"$a(x)$"),
    ("muK", r"$\mu_K(x)$ [MeV]"),
    ("T", r"$T(x)$ [MeV]"),
    ("u", r"$u(x)$"),
]

x_ref = reference_profile["x"].to_numpy(dtype=float) / 1.0e6
x_new = new_profile["x"].to_numpy(dtype=float) / 1.0e6

fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.8), constrained_layout=True)
for ax, (column, ylabel) in zip(axes.ravel(), plot_specs):
    ax.plot(
        x_ref,
        reference_profile[column],
        color="0.15",
        lw=2.4,
        ls="--",
        label="saved CSV profile",
        zorder=3,
    )
    ax.plot(
        x_new,
        new_profile[column],
        color="tab:blue",
        lw=1.6,
        alpha=0.78,
        label="updated solver",
        zorder=2,
    )
    if column in reference_endpoint.columns and not reference_endpoint.empty:
        ax.scatter(
            reference_endpoint["x"].to_numpy(dtype=float) / 1.0e6,
            reference_endpoint[column].to_numpy(dtype=float),
            s=40,
            facecolors="none",
            edgecolors="tab:red",
            linewidths=1.2,
            label="saved endpoint row",
            zorder=4,
        )
    ax.set_xlabel(r"$x$ [$10^6\,\mathrm{MeV}^{-1}$]")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(loc="best", frameon=True, fontsize=8)

fig.suptitle("Energy-conserving profile comparison", y=1.02)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.2), constrained_layout=True)
for ax, (column, ylabel) in zip(axes.ravel(), plot_specs):
    ref_values = reference_profile[column].to_numpy(dtype=float)
    new_on_ref_x = np.interp(reference_profile["x"].to_numpy(dtype=float), new_profile["x"].to_numpy(dtype=float), new_profile[column].to_numpy(dtype=float))
    ax.plot(x_ref, new_on_ref_x - ref_values, color="tab:purple", lw=1.8, label="updated - saved")
    ax.axhline(0.0, color="0.45", lw=1.0, ls="--")
    ax.set_xlabel(r"$x$ [$10^6\,\mathrm{MeV}^{-1}$]")
    ax.set_ylabel("difference in " + ylabel)
    ax.grid(alpha=0.25)
    ax.legend(loc="best", frameon=True, fontsize=8)

fig.suptitle("Pointwise disagreement after interpolating updated solver onto saved CSV x-grid", y=1.02)
plt.show()


## Profile Residue Inputs

Contains the `PROFILE_RESIDUE_INPUTS` notebook section.


In [ ]:
profiles_for_residue = {
    "saved CSV profile": reference_all,
    "updated full solver": new_profile,
}


def _require_columns(df, label, columns):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise RuntimeError(f"{label} profile is missing required columns: {missing}")


def _dx(values, x):
    values = np.asarray(values, dtype=float)
    x = np.asarray(x, dtype=float)
    if values.size < 3:
        raise RuntimeError("Need at least three quark-side profile rows for edge_order=2 derivatives")
    return np.gradient(values, x, edge_order=2)


def _residue_over_profile(residue, comparison_profile):
    residue = np.asarray(residue, dtype=float)
    comparison_profile = np.asarray(comparison_profile, dtype=float)
    finite = np.abs(comparison_profile[np.isfinite(comparison_profile)])
    floor = max(float(np.nanmedian(finite)) if finite.size else 0.0, np.finfo(float).tiny)
    return residue / np.maximum(np.abs(comparison_profile), floor)


def _residue_over_scalar(residue, scalar_scale):
    residue = np.asarray(residue, dtype=float)
    scalar_scale = float(scalar_scale)
    if (not np.isfinite(scalar_scale)) or abs(scalar_scale) <= np.finfo(float).tiny:
        return np.full_like(residue, np.nan, dtype=float)
    return residue / scalar_scale


def _stats(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            "max_abs": np.nan,
            "rms": np.nan,
            "median_abs": np.nan,
            "p95_abs": np.nan,
            "interior_max_abs": np.nan,
        }
    abs_values = np.abs(finite)
    if values.size > 4:
        interior = values[2:-2]
        interior = interior[np.isfinite(interior)]
    else:
        interior = finite
    return {
        "max_abs": float(np.max(abs_values)),
        "rms": float(np.sqrt(np.mean(finite**2))),
        "median_abs": float(np.median(abs_values)),
        "p95_abs": float(np.percentile(abs_values, 95.0)),
        "interior_max_abs": float(np.max(np.abs(interior))) if interior.size else np.nan,
    }


def compute_profile_residues(label, df):
    base_columns = ["x", "a", "u", "P", "h", "nB", "D", "gamma", "eta"]
    _require_columns(df, label, base_columns)

    qdf = df.loc[np.isfinite(df["D"])].copy().reset_index(drop=True)
    x = qdf["x"].to_numpy(dtype=float)
    if not np.all(np.isfinite(x)) or not np.all(np.diff(x) > 0.0):
        raise RuntimeError(f"{label} quark-side x values must be finite and strictly increasing")

    a = qdf["a"].to_numpy(dtype=float)
    u = qdf["u"].to_numpy(dtype=float)
    nB = qdf["nB"].to_numpy(dtype=float)
    h = qdf["h"].to_numpy(dtype=float)
    P = qdf["P"].to_numpy(dtype=float)
    D = qdf["D"].to_numpy(dtype=float)
    gamma_rate = qdf["gamma"].to_numpy(dtype=float)
    eta = qdf["eta"].to_numpy(dtype=float)

    if not np.all(np.isfinite(D)) or not np.all(np.isfinite(gamma_rate)) or not np.all(np.isfinite(eta)):
        raise RuntimeError(f"{label} has non-finite D/gamma/eta values on quark-side rows")

    da_dx = _dx(a, x)
    q_profile = qdf["q"].to_numpy(dtype=float) if "q" in qdf.columns else D * da_dx - u * a
    q_prime = _dx(q_profile, x)
    jB_profile = u * nB
    Pi_profile = h * u**2 + P
    E_profile = h * u * np.sqrt(1.0 + u**2)
    a_mean = float(np.nanmean(a))
    R_profile = gamma_rate * (a**3 + eta * a)
    R_at_mean_a_profile = gamma_rate * (a_mean**3 + eta * a_mean)
    R_at_mean_a = float(np.nanmean(R_at_mean_a_profile))
    kaon_equation_raw = q_prime - R_profile
    kaon_equation_mean_residual = float(np.nanmean(kaon_equation_raw))
    kaon_equation_mean_residue = (
        kaon_equation_mean_residual / R_at_mean_a
        if np.isfinite(R_at_mean_a) and abs(R_at_mean_a) > np.finfo(float).tiny
        else np.nan
    )

    raw = {
        "r_jB": _dx(jB_profile, x),
        "r_Pi": _dx(Pi_profile, x),
        "r_KaonNess": kaon_equation_raw,
        "r_E": _dx(E_profile, x),
    }
    residue = {
        "r_jB": _residue_over_profile(raw["r_jB"], jB_profile),
        "r_Pi": _residue_over_profile(raw["r_Pi"], Pi_profile),
        "r_KaonNess": _residue_over_scalar(raw["r_KaonNess"], R_at_mean_a),
        "r_E": _residue_over_profile(raw["r_E"], E_profile),
    }
    comparison_profiles = {
        "r_jB": jB_profile,
        "r_Pi": Pi_profile,
        "r_KaonNess": np.full_like(a, R_at_mean_a, dtype=float),
        "r_E": E_profile,
    }
    derived = {
        "jB_profile": jB_profile,
        "Pi_profile": Pi_profile,
        "E_profile": E_profile,
        "q_profile": q_profile,
        "q_prime": q_prime,
        "R_profile": R_profile,
        "R_at_mean_a_profile": R_at_mean_a_profile,
        "R_at_mean_a": R_at_mean_a,
        "a_mean_for_R_normalization": a_mean,
        "kaon_equation_mean_residual": kaon_equation_mean_residual,
        "kaon_equation_mean_residue": kaon_equation_mean_residue,
        "da_dx": da_dx,
    }

    return {
        "label": label,
        "profile": qdf,
        "x": x,
        "raw": raw,
        "residue": residue,
        "comparison_profiles": comparison_profiles,
        "derived": derived,
    }


residue_results = {
    label: compute_profile_residues(label, df)
    for label, df in profiles_for_residue.items()
}

summary_rows = []
for label, residue_result in residue_results.items():
    for residue_name, values in residue_result["residue"].items():
        row = {"profile": label, "residue": residue_name}
        row.update(_stats(values))
        if residue_name == "r_KaonNess":
            row.update(
                {
                    "mean_raw_residual": residue_result["derived"]["kaon_equation_mean_residual"],
                    "R_at_mean_a": residue_result["derived"]["R_at_mean_a"],
                    "mean_residual_over_R_mean_a": residue_result["derived"]["kaon_equation_mean_residue"],
                    "mean_a": residue_result["derived"]["a_mean_for_R_normalization"],
                }
            )
        summary_rows.append(row)

residue_summary_df = pd.DataFrame(summary_rows)
display(residue_summary_df)


## Profile Residue Plotter

Visualizes the outputs from the `PROFILE_RESIDUE_PLOTTER` workflow section.


In [ ]:
def plot_profile_residues(residue_result):
    label = residue_result["label"]
    x = residue_result["x"] / 1.0e6
    residues = residue_result["residue"]

    fig, axes = plt.subplots(
        len(residues),
        1,
        figsize=(9.5, max(2.2 * len(residues), 5.5)),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)

    for ax, (residue_name, values) in zip(axes, residues.items()):
        ax.scatter(x, np.abs(values), s=16, label=residue_name)
        ax.set_yscale("linear")
        ax.set_ylabel(f"|{residue_name}|")
        ax.axvline(0.0, color="0.65", lw=1.0, ls="--")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(loc="best", frameon=True, fontsize=8)

    axes[0].set_title(f"{label}: residue diagnostics; kaon row uses scalar R(mean(a))")
    axes[-1].set_xlabel(r"$x$ [$10^6\,\mathrm{MeV}^{-1}$]")
    return fig


for residue_result in residue_results.values():
    plot_profile_residues(residue_result)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(10.0, 7.2), sharex=False, constrained_layout=True)
for label, residue_result in residue_results.items():
    x = residue_result["x"] / 1.0e6
    kaon_residue = residue_result["residue"]["r_KaonNess"]
    running_mean = np.cumsum(residue_result["raw"]["r_KaonNess"]) / np.arange(1, kaon_residue.size + 1)
    R_at_mean_a = residue_result["derived"]["R_at_mean_a"]
    running_mean_norm = _residue_over_scalar(running_mean, R_at_mean_a)
    axes[0].plot(x, kaon_residue, lw=1.8, label=label)
    axes[1].plot(x, running_mean_norm, lw=1.8, label=label)

axes[0].axhline(0.0, color="0.5", lw=1.0, ls="--")
axes[0].set_ylabel(r"$(q' - R(a)) / R(\langle a\rangle)$")
axes[0].set_title("Pointwise kaon-equation residue with scalar R(mean(a)) normalization")
axes[0].grid(alpha=0.25)
axes[0].legend(loc="best", frameon=True, fontsize=8)

axes[1].axhline(0.0, color="0.5", lw=1.0, ls="--")
axes[1].set_xlabel(r"$x$ [$10^6\,\mathrm{MeV}^{-1}$]")
axes[1].set_ylabel(r"running mean / $R(\langle a\rangle)$")
axes[1].set_title("Cumulative mean of q' - R(a), compared with saved CSV")
axes[1].grid(alpha=0.25)
axes[1].legend(loc="best", frameon=True, fontsize=8)
plt.show()

kaon_comparison_rows = []
for label, residue_result in residue_results.items():
    derived = residue_result["derived"]
    kaon_comparison_rows.append(
        {
            "profile": label,
            "mean_a": derived["a_mean_for_R_normalization"],
            "R_at_mean_a": derived["R_at_mean_a"],
            "mean_qprime_minus_R": derived["kaon_equation_mean_residual"],
            "mean_residual_over_R_mean_a": derived["kaon_equation_mean_residue"],
            "max_abs_pointwise_over_R_mean_a": float(np.nanmax(np.abs(residue_result["residue"]["r_KaonNess"]))),
            "rms_pointwise_over_R_mean_a": float(np.sqrt(np.nanmean(residue_result["residue"]["r_KaonNess"]**2))),
        }
    )

kaon_comparison_df = pd.DataFrame(kaon_comparison_rows)
display(kaon_comparison_df)


## Profile Exports

Contains the `PROFILE_EXPORTS` notebook section.


In [ ]:
output_dir = Path("csv_share")
output_dir.mkdir(exist_ok=True)

latest_profile_path = output_dir / "energy_conserving_full_solver_profiles_T10MeV_B14_180MeV_nBN_3n0_aQstar_0p3.csv"
latest_profile = new_profile.copy()
latest_profile["dummy"] = 0.0
latest_profile["e"] = latest_profile["h"] - latest_profile["P"]
latest_profile["nK"] = latest_profile["a"] * latest_profile["nB"]
latest_profile["P_over_e"] = latest_profile["P"] / latest_profile["e"]

save_columns = [
    "dummy",
    "x",
    "a",
    "q",
    "u",
    "P",
    "e",
    "h",
    "nB",
    "nK",
    "muB",
    "muK",
    "T",
    "h_over_nB",
    "P_over_e",
    "D",
    "gamma",
    "eta",
]
save_columns = [column for column in save_columns if column in latest_profile.columns]
latest_profile = latest_profile[save_columns]
latest_profile.to_csv(latest_profile_path, index=False)

display(
    pd.DataFrame(
        [
            {
                "saved_csv": str(latest_profile_path),
                "rows": int(latest_profile.shape[0]),
                "columns": int(latest_profile.shape[1]),
                "first_x": float(latest_profile["x"].iloc[0]),
                "last_x": float(latest_profile["x"].iloc[-1]),
            }
        ]
    )
)
